In [16]:
# ติดตั้ง geemap สำหรับแสดงแผนที่ใน Colab
!pip install geemap -q
import geemap

In [17]:
import ee
ee.Authenticate()
ee.Initialize(project='ee-jutharatview')

chiangrai = (
    ee.FeatureCollection("FAO/GAUL/2015/level1")
    .filter(ee.Filter.eq("ADM1_NAME", "Chiang Rai"))
)
aoi = chiangrai.geometry()

rivers = (
    ee.FeatureCollection("WWF/HydroSHEDS/v1/FreeFlowingRivers")
    .filterBounds(aoi)
)
river_raster = (
    rivers.reduceToImage(properties=["RIV_ORD"], reducer=ee.Reducer.min())
          .unmask(0).gt(0).clip(aoi)
)
dist_river = river_raster.Not().cumulativeCost(
    source=river_raster, maxDistance=50000
).clip(aoi)

# เช็ค band name
print(dist_river.bandNames().getInfo())

['cumulative_cost']


In [18]:
import pandas as pd

framework = pd.DataFrame({
    "Factor":      ["Slope", "Rainfall (max monthly)", "Land cover", "Distance to river"],
    "Data Source": ["SRTM DEM (USGS/SRTMGL1_003)",
                    "ERA5 Monthly (ECMWF/ERA5_LAND)",
                    "MODIS MCD12Q1",
                    "HydroSHEDS FreeFlowingRivers"],
    "Weight":      [0.30, 0.30, 0.20, 0.20],
    "Direction":   ["↑ ชันยิ่งเสี่ยง", "↑ ฝนมากยิ่งเสี่ยง",
                    "Urban=1.0, Forest=0.1", "↓ ใกล้แม่น้ำยิ่งเสี่ยง"],
    "Reference":   ["Costache et al. 2020", "Tehrany et al. 2015",
                    "Gayen et al. 2019", "Tehrany et al. 2015"]
})

print("=" * 60)
print("Flash Flood Risk Model — Chiang Rai")
print("Research Question: พื้นที่ใดมีความเสี่ยงน้ำท่วมฉับพลันสูง?")
print("=" * 60)
print(f"\nWLC Equation:")
print("Risk = 0.30·Slope + 0.30·Rainfall + 0.20·LandCover + 0.20·DistRiver")
print(f"\nWeight sum = {0.30+0.30+0.20+0.20:.2f} ✅")
print()
display(framework)

Flash Flood Risk Model — Chiang Rai
Research Question: พื้นที่ใดมีความเสี่ยงน้ำท่วมฉับพลันสูง?

WLC Equation:
Risk = 0.30·Slope + 0.30·Rainfall + 0.20·LandCover + 0.20·DistRiver

Weight sum = 1.00 ✅



,Factor,Data Source,Weight,Direction,Reference
0,Slope,SRTM DEM (USGS/SRTMGL1_003),0.3,↑ ชันยิ่งเสี่ยง,Costache et al. 2020
1,Rainfall (max monthly),ERA5 Monthly (ECMWF/ERA5_LAND),0.3,↑ ฝนมากยิ่งเสี่ยง,Tehrany et al. 2015
2,Land cover,MODIS MCD12Q1,0.2,"Urban=1.0, Forest=0.1",Gayen et al. 2019
3,Distance to river,HydroSHEDS FreeFlowingRivers,0.2,↓ ใกล้แม่น้ำยิ่งเสี่ยง,Tehrany et al. 2015


In [19]:
# ใช้ JRC เป็น "label" สำหรับ training
# Flood = 1 (occurrence >= 20%), No flood = 0

jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").clip(aoi)

flood_mask    = jrc.gte(20)   # พื้นที่ท่วม
no_flood_mask = jrc.lt(5)     # พื้นที่ไม่ท่วมชัดเจน (หลีกเลี่ยง 5-19% ที่ ambiguous)

# Sample points: 500 flood + 500 no-flood
flood_points = flood_mask.selfMask().sample(
    region     = aoi,
    scale      = 500,
    numPixels  = 500,
    seed       = 42,
    geometries = True
).map(lambda f: f.set("flood", 1))

no_flood_points = no_flood_mask.selfMask().sample(
    region     = aoi,
    scale      = 500,
    numPixels  = 500,
    seed       = 42,
    geometries = True
).map(lambda f: f.set("flood", 0))

samples = flood_points.merge(no_flood_points)
print(f"Total samples: {samples.size().getInfo()}")

Total samples: 60


In [20]:
dem = ee.Image("USGS/SRTMGL1_003").clip(aoi)

slope = ee.Terrain.slope(dem).rename("slope")

In [21]:
rainfall = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY")
    .filterBounds(aoi)
    .filterDate("2020-01-01", "2020-12-31")
    .select("total_precipitation")
)

rainfall_mm = rainfall.sum().rename("total_precipitation_sum").clip(aoi)

In [22]:
lc = ee.Image("MODIS/061/MCD12Q1/2020_01_01").select("LC_Type1").clip(aoi)

lc_risk = (
    lc.remap(
        [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16],
        [0.1,0.1,0.1,0.1,0.2,0.2,0.3,0.3,0.3,0.5,0.5,0.7,1.0,0.8,0.2,0.1,0.1]
    ).rename("lc_risk")
)

In [23]:
print(slope.bandNames().getInfo())
print(rainfall_mm.bandNames().getInfo())
print(lc_risk.bandNames().getInfo())

['slope']
['total_precipitation_sum']
['lc_risk']


In [24]:
# Stack ปัจจัยทั้ง 4 เป็น multiband image
# ใช้ raw values (ไม่ต้อง normalize เพราะ RF จัดการ scale เองได้)

feature_stack = (
     slope                           # band: slope
    .addBands(rainfall_mm)          # band: total_precipitation_sum
    .addBands(lc_risk.rename("lc_risk"))   # band: lc_risk
    .addBands(dist_river.rename("dist_river"))  # band: cumulative_cost
    .clip(aoi)
)

FEATURE_BANDS = ["slope", "total_precipitation_sum", "lc_risk", "dist_river"]
# Extract feature values ที่ sample points
training_data = feature_stack.select(FEATURE_BANDS).sampleRegions(
    collection  = samples,
    properties  = ["flood"],
    scale       = 500,
    tileScale   = 8
)

# แบ่ง train/test 80:20
training_data = training_data.randomColumn(seed=42)
train = training_data.filter(ee.Filter.lt("random", 0.8))
test  = training_data.filter(ee.Filter.gte("random", 0.8))

print(f"Train: {train.size().getInfo()} samples")
print(f"Test : {test.size().getInfo()} samples")

Train: 48 samples
Test : 12 samples


In [25]:
# Train Random Forest Classifier
rf_model = ee.Classifier.smileRandomForest(
    numberOfTrees         = 100,   # จำนวนต้นไม้
    variablesPerSplit     = 2,     # sqrt(4 features) ≈ 2
    minLeafPopulation     = 5,
    bagFraction           = 0.632, # standard bootstrap fraction
    seed                  = 42
).train(
    features           = train,
    classProperty      = "flood",
    inputProperties    = FEATURE_BANDS
)

print("✅ Random Forest trained")

# Feature Importance
importance = rf_model.explain().get("importance").getInfo()
print("\nFeature Importance:")
for feat, score in sorted(importance.items(), key=lambda x: -x[1]):
    bar = "█" * int(score / max(importance.values()) * 30)
    print(f"  {feat:<35} {score:.4f}  {bar}")

✅ Random Forest trained

Feature Importance:
  slope                               8.6413  ██████████████████████████████
  total_precipitation_sum             4.9636  █████████████████
  dist_river                          4.6949  ████████████████
  lc_risk                             1.0200  ███


In [26]:
# ── Classify ทั้งพื้นที่ ──────────────────────────────────────────────
# เปลี่ยนเป็น probability mode ก่อน
rf_prob = rf_model.setOutputMode("PROBABILITY")

flood_prob = feature_stack.select(FEATURE_BANDS).classify(rf_prob).rename("flood_prob").clip(aoi)
flood_class = flood_prob.gte(0.50).rename("flood_class")  # threshold 0.5

# ── Validate บน test set ──────────────────────────────────────────────
test_classified = test.classify(rf_model)

cm = test_classified.errorMatrix(
    actual    = "flood",
    predicted = "classification"
)

print("=" * 50)
print("Random Forest — Validation (test set 20%)")
print("=" * 50)
print(f"\nConfusion Matrix:\n{cm.array().getInfo()}")
print(f"\nOverall Accuracy : {cm.accuracy().getInfo():.3f}")
print(f"Kappa Coefficient: {cm.kappa().getInfo():.3f}")

# Producer's / Consumer's Accuracy
producers = cm.producersAccuracy().getInfo()
consumers = cm.consumersAccuracy().getInfo()
print(f"\nProducer Accuracy (Recall):")
print(f"  No flood : {producers[0][0]:.3f}")
print(f"  Flood    : {producers[1][0]:.3f}")
print(f"\nConsumer Accuracy (Precision):")
print(f"  No flood : {consumers[0][0]:.3f}")
print(f"  Flood    : {consumers[0][1]:.3f}")

# ── แผนที่ ────────────────────────────────────────────────────────────
Map4 = geemap.Map()
Map4.centerObject(aoi, 9)
Map4.addLayer(
    flood_prob,
    {"min": 0, "max": 1, "palette": ["#1a9850", "#fee08b", "#f46d43", "#a50026"]},
    "RF Flood Probability"
)
Map4.addLayer(
    flood_class.selfMask(),
    {"palette": ["#d62728"]},
    "RF Predicted Flood (prob >= 0.5)"
)
Map4.addLayer(chiangrai, {"color": "black"}, "Boundary")
Map4

Random Forest — Validation (test set 20%)

Confusion Matrix:
[[1, 2], [4, 5]]

Overall Accuracy : 0.500
Kappa Coefficient: -0.091

Producer Accuracy (Recall):
  No flood : 0.333
  Flood    : 0.556

Consumer Accuracy (Precision):
  No flood : 0.200
  Flood    : 0.714


Map(center=[19.845006011670804, 99.8693349073738], controls=(WidgetControl(options=['position', 'transparent_b…

In [27]:
rain_stats = rainfall_mm.reduceRegion(
    reducer=ee.Reducer.minMax(), geometry=aoi, scale=10000, maxPixels=1e9
)
rain_min = rain_stats.getNumber("total_precipitation_sum_min")
rain_max = rain_stats.getNumber("total_precipitation_sum_max")
rainfall_norm = (
    rainfall_mm.subtract(rain_min)
               .divide(rain_max.subtract(rain_min))
               .clamp(0, 1).rename("rainfall_norm")
)

slope_norm = slope.divide(60).clamp(0, 1).rename("slope_norm")
lc_norm    = lc_risk.rename("lc_norm")

dist_max = dist_river.reduceRegion(
    reducer=ee.Reducer.max(), geometry=aoi, scale=500, maxPixels=1e9
).getNumber("cumulative_cost")
dist_norm = ee.Image(1).subtract(dist_river.divide(dist_max)).clamp(0, 1).rename("dist_norm")

flood_risk = (
    slope_norm.multiply(0.30)
    .add(rainfall_norm.multiply(0.30))
    .add(lc_norm.multiply(0.20))
    .add(dist_norm.multiply(0.20))
    .rename("flood_risk").clip(aoi)
)
classified = (
    flood_risk
    .where(flood_risk.lt(0.25), 1)
    .where(flood_risk.gte(0.25).And(flood_risk.lt(0.50)), 2)
    .where(flood_risk.gte(0.50).And(flood_risk.lt(0.75)), 3)
    .where(flood_risk.gte(0.75), 4)
    .rename("risk_class").clip(aoi)
)

print("✅ classified พร้อมแล้ว")

✅ classified พร้อมแล้ว


In [ ]:
# เปรียบเทียบพื้นที่ที่สองโมเดลเห็นต่างกัน
wlc_high = classified.gte(3)      # WLC บอกว่าเสี่ยงสูง
rf_high  = flood_class            # RF บอกว่าเสี่ยงสูง

agree_high  = wlc_high.eq(1).And(rf_high.eq(1))   # ทั้งคู่บอกสูง
agree_low   = wlc_high.eq(0).And(rf_high.eq(0))   # ทั้งคู่บอกต่ำ
only_wlc    = wlc_high.eq(1).And(rf_high.eq(0))   # WLC สูง RF ต่ำ
only_rf     = wlc_high.eq(0).And(rf_high.eq(1))   # RF สูง WLC ต่ำ

def area(img):
    return (img.multiply(ee.Image.pixelArea())
               .reduceRegion(ee.Reducer.sum(), aoi, 500, maxPixels=1e9)
               .values().get(0).getInfo()) / 1e6

print("=" * 45)
print("WLC vs Random Forest — Agreement")
print("=" * 45)
print(f"  Both HIGH (agree)     : {area(agree_high):8.1f} km²")
print(f"  Both LOW  (agree)     : {area(agree_low):8.1f} km²")
print(f"  Only WLC = High       : {area(only_wlc):8.1f} km²  ← WLC overpredicts?")
print(f"  Only RF  = High       : {area(only_rf):8.1f} km²  ← RF จับ pattern เพิ่ม?")

Map5 = geemap.Map()
Map5.centerObject(aoi, 9)
Map5.addLayer(agree_high.selfMask(),  {"palette": ["#238b45"]}, "Both HIGH")
Map5.addLayer(only_wlc.selfMask(),    {"palette": ["#f46d43"]}, "Only WLC High")
Map5.addLayer(only_rf.selfMask(),     {"palette": ["#7b2d8b"]}, "Only RF High")
Map5.addLayer(chiangrai, {"color": "black"}, "Boundary")
Map5.add_legend(title="Model Agreement", legend_dict={
    "Both agree HIGH": "#238b45",
    "Only WLC = High": "#f46d43",
    "Only RF = High":  "#7b2d8b"
})
Map5

WLC vs Random Forest — Agreement
  Both HIGH (agree)     :   1199.7 km²
  Both LOW  (agree)     :    311.7 km²
  Only WLC = High       :      0.0 km²  ← WLC overpredicts?
  Only RF  = High       :   9780.1 km²  ← RF จับ pattern เพิ่ม?


Map(center=[19.845006011670804, 99.8693349073738], controls=(WidgetControl(options=['position', 'transparent_b…

In [29]:
# ── Export RF Flood Probability ───────────────────────────────────────
task1 = ee.batch.Export.image.toDrive(
    image          = flood_prob,
    description    = "RF_FloodProbability_ChiangRai1",
    folder         = "GEE_Lab4",
    fileNamePrefix = "rf_flood_probability",
    region         = aoi,
    scale          = 500,
    crs            = "EPSG:4326",
    maxPixels      = 1e9
)
task1.start()
print(f"✅ Task 1 started: RF Flood Probability — {task1.id}")

# ── Export RF Flood Class (0/1) ───────────────────────────────────────
task2 = ee.batch.Export.image.toDrive(
    image          = flood_class.toByte(),
    description    = "RF_FloodClass_ChiangRai1",
    folder         = "GEE_Lab4",
    fileNamePrefix = "rf_flood_class",
    region         = aoi,
    scale          = 500,
    crs            = "EPSG:4326",
    maxPixels      = 1e9
)
task2.start()
print(f"✅ Task 2 started: RF Flood Class — {task2.id}")

# ── Export WLC Classified ─────────────────────────────────────────────
task3 = ee.batch.Export.image.toDrive(
    image          = classified.toByte(),
    description    = "WLC_FloodRisk_ChiangRai1",
    folder         = "GEE_Lab4",
    fileNamePrefix = "wlc_flood_classified",
    region         = aoi,
    scale          = 500,
    crs            = "EPSG:4326",
    maxPixels      = 1e9
)
task3.start()
print(f"✅ Task 3 started: WLC Classified — {task3.id}")

# ── ตรวจสอบสถานะ ──────────────────────────────────────────────────────
import time
print("\nรอสักครู่แล้วเช็คสถานะ...")
time.sleep(5)

for task in [task1, task2, task3]:
    status = task.status()
    print(f"  {status['description']}: {status['state']}")

print("\nดูสถานะ export ได้ที่ https://code.earthengine.google.com/tasks")
print("ไฟล์จะอยู่ใน Google Drive โฟลเดอร์ GEE_Lab4 ครับ")

✅ Task 1 started: RF Flood Probability — ZMMT4XXOV62OPW6SL4NX3OCB
✅ Task 2 started: RF Flood Class — MTI64LF5PY7I3MB4ZXRR5PF5
✅ Task 3 started: WLC Classified — 6J2GLCZRUKTGUHPIPRSU5QX3

รอสักครู่แล้วเช็คสถานะ...
  RF_FloodProbability_ChiangRai1: READY
  RF_FloodClass_ChiangRai1: READY
  WLC_FloodRisk_ChiangRai1: READY

ดูสถานะ export ได้ที่ https://code.earthengine.google.com/tasks
ไฟล์จะอยู่ใน Google Drive โฟลเดอร์ GEE_Lab4 ครับ
